# Customer Retention & Revenue Intelligence

## Phase 1: Raw Data Profiling

### Objective
Assess the structure, completeness, consistency, and quality of the raw customer, usage, and churn datasets before performing any cleaning or transformation.

### Raw Datasets
- `customer_info.csv` — customer demographics, contract information, monthly charges, and signup dates
- `usage_data.csv` — monthly customer usage and complaint activity
- `churn_labels.csv` — customer churn status

In [1]:
import pandas as pd

In [2]:
customers = pd.read_csv("../data/raw/customer_info.csv")

In [4]:
customers.head()

,CustomerID,Age,Gender,Region,ContractType,MonthlyCharges,SignupDate
0,CUST_0,NaN,Male,Urban,Yearly,48.18,2021-03-20
1,CUST_1,NaN,Female,Suburban,Prepaid,38.94,2020-01-25
2,CUST_2,46.0,Female,Rural,Monthly,48.56,2022-08-31
3,CUST_3,NaN,Female,Urban,Prepaid,24.22,2018-12-09
4,CUST_4,60.0,Male,Rural,Monthly,67.86,2022-12-28


In [5]:
customers.shape

(10200, 7)

In [6]:
customers.info()

<class 'pandas.DataFrame'>
RangeIndex: 10200 entries, 0 to 10199
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   CustomerID      10200 non-null  str    
 1   Age             6617 non-null   float64
 2   Gender          10200 non-null  str    
 3   Region          10200 non-null  str    
 4   ContractType    10200 non-null  str    
 5   MonthlyCharges  10200 non-null  float64
 6   SignupDate      10200 non-null  str    
dtypes: float64(2), str(5)
memory usage: 557.9 KB


In [7]:
customers.isnull().sum()

CustomerID           0
Age               3583
Gender               0
Region               0
ContractType         0
MonthlyCharges       0
SignupDate           0
dtype: int64

In [8]:
customers.duplicated().sum()

np.int64(195)

In [9]:
customers.describe(include="all")

,CustomerID,Age,Gender,Region,ContractType,MonthlyCharges,SignupDate
count,10200,6617.000000,10200,10200,10200,10200.000000,10200
unique,10000,NaN,2,3,3,NaN,1995
top,CUST_171,NaN,Female,Suburban,Prepaid,NaN,2019-03-02
freq,2,NaN,5117,3479,4117,NaN,14
mean,NaN,43.424664,NaN,NaN,NaN,50.961995,NaN
std,NaN,14.966108,NaN,NaN,NaN,21.423985,NaN
min,NaN,18.000000,NaN,NaN,NaN,-6.720000,NaN
25%,NaN,31.000000,NaN,NaN,NaN,39.767500,NaN
50%,NaN,43.000000,NaN,NaN,NaN,50.000000,NaN
75%,NaN,56.000000,NaN,NaN,NaN,60.460000,NaN


In [10]:
customers["CustomerID"].duplicated().sum()

np.int64(200)

In [11]:
customers["CustomerID"].nunique()

10000

In [12]:
customers["Age"].describe()

count    6617.000000
mean       43.424664
std        14.966108
min        18.000000
25%        31.000000
50%        43.000000
75%        56.000000
max        69.000000
Name: Age, dtype: float64

In [13]:
for column in ["Gender", "Region", "ContractType"]:
    print(f"\n{column}:")
    print(customers[column].value_counts(dropna=False))


Gender:
Gender
Female    5117
Male      5083
Name: count, dtype: int64

Region:
Region
Suburban    3479
Rural       3363
Urban       3358
Name: count, dtype: int64

ContractType:
ContractType
Prepaid    4117
Monthly    4040
Yearly     2043
Name: count, dtype: int64


In [14]:
duplicate_ids = customers[
    customers["CustomerID"].duplicated(keep=False)
].sort_values("CustomerID")

duplicate_ids

,CustomerID,Age,Gender,Region,ContractType,MonthlyCharges,SignupDate
1031,CUST_1031,57.0,Male,Urban,Monthly,40.60,2023-05-23
10070,CUST_1031,57.0,Male,Urban,Monthly,40.60,2023-05-23
1042,CUST_1042,NaN,Male,Urban,Monthly,63.09,2021-04-24
10075,CUST_1042,NaN,Male,Urban,Monthly,63.09,2021-04-24
10061,CUST_1128,55.0,Male,Urban,Prepaid,31.88,2019-01-18
...,...,...,...,...,...,...,...
10111,CUST_9881,36.0,Male,Suburban,Yearly,56.38,2018-02-04
10096,CUST_9977,NaN,Male,Urban,Monthly,34.32,2020-03-22
9977,CUST_9977,NaN,Male,Urban,Monthly,34.32,2020-03-22
999,CUST_999,55.0,Female,Urban,Prepaid,37.78,2021-04-16


In [15]:
conflicting_ids = (
    duplicate_ids
    .groupby("CustomerID")
    .filter(lambda x: len(x.drop_duplicates()) > 1)
)

conflicting_ids

,CustomerID,Age,Gender,Region,ContractType,MonthlyCharges,SignupDate
10024,CUST_248,22.0,Male,Rural,Monthly,58.85,2022-06-09
248,CUST_248,22.0,Male,Rural,Monthly,58.85,2022-09-06
10144,CUST_308,37.0,Male,Urban,Monthly,42.28,2018-03-01
308,CUST_308,37.0,Male,Urban,Monthly,42.28,2018-01-03
10089,CUST_333,NaN,Male,Rural,Prepaid,63.64,2023-02-11
333,CUST_333,NaN,Male,Rural,Prepaid,63.64,2023-11-02
391,CUST_391,NaN,Female,Urban,Yearly,31.60,2023-11-03
10150,CUST_391,NaN,Female,Urban,Yearly,31.60,2023-03-11
466,CUST_466,23.0,Female,Urban,Monthly,45.18,2021-03-07
10076,CUST_466,23.0,Female,Urban,Monthly,45.18,2021-07-03


In [16]:
usage = pd.read_csv("../data/raw/usage_data.csv")

usage.head()

,CustomerID,Month,CallMinutes,DataUsageGB,SMSCount,Complaints
0,CUST_0,2023-01-31,518.12,7.21,9,1
1,CUST_0,2023-02-28,439.79,5.99,35,1
2,CUST_0,2023-03-31,545.06,14.16,14,1
3,CUST_0,2023-04-30,461.78,6.66,40,0
4,CUST_0,2023-05-31,434.82,12.75,26,2


In [17]:
usage.shape

(120000, 6)

In [18]:
usage.info()

<class 'pandas.DataFrame'>
RangeIndex: 120000 entries, 0 to 119999
Data columns (total 6 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   CustomerID   120000 non-null  str    
 1   Month        120000 non-null  str    
 2   CallMinutes  120000 non-null  float64
 3   DataUsageGB  120000 non-null  float64
 4   SMSCount     120000 non-null  int64  
 5   Complaints   120000 non-null  int64  
dtypes: float64(2), int64(2), str(2)
memory usage: 5.5 MB


In [19]:
usage.isnull().sum()

CustomerID     0
Month          0
CallMinutes    0
DataUsageGB    0
SMSCount       0
Complaints     0
dtype: int64

In [20]:
usage.duplicated().sum()

np.int64(0)

In [21]:
usage.describe(include="all")

,CustomerID,Month,CallMinutes,DataUsageGB,SMSCount,Complaints
count,120000,120000,120000.000000,120000.000000,120000.000000,120000.000000
unique,10000,12,NaN,NaN,NaN,NaN
top,CUST_0,2023-01-31,NaN,NaN,NaN,NaN
freq,12,10000,NaN,NaN,NaN,NaN
mean,NaN,NaN,492.401958,9.898117,24.525392,0.503000
std,NaN,NaN,117.443986,3.067815,14.423944,0.708763
min,NaN,NaN,34.140000,0.000000,0.000000,0.000000
25%,NaN,NaN,418.450000,7.810000,12.000000,0.000000
50%,NaN,NaN,494.960000,9.920000,25.000000,0.000000
75%,NaN,NaN,571.080000,11.980000,37.000000,1.000000


In [22]:
months_per_customer = usage.groupby("CustomerID")["Month"].nunique()

months_per_customer.value_counts()

Month
12    10000
Name: count, dtype: int64

In [23]:
usage.duplicated(subset=["CustomerID", "Month"]).sum()

np.int64(0)

In [24]:
churn = pd.read_csv("../data/raw/churn_labels.csv")

churn.head()

,CustomerID,Churn
0,CUST_0,0
1,CUST_1,0
2,CUST_2,0
3,CUST_3,0
4,CUST_4,0


In [25]:
churn.shape

(10000, 2)

In [26]:
churn.isnull().sum()

CustomerID    0
Churn         0
dtype: int64

In [27]:
churn.duplicated().sum()

np.int64(0)

In [28]:
churn["CustomerID"].duplicated().sum()

np.int64(0)

In [29]:
churn["Churn"].value_counts(dropna=False)

Churn
0    9214
1     786
Name: count, dtype: int64

In [30]:
months_per_customer = usage.groupby("CustomerID")["Month"].nunique()
months_per_customer.value_counts()

Month
12    10000
Name: count, dtype: int64

In [31]:
usage.duplicated(subset=["CustomerID", "Month"]).sum()

np.int64(0)

## Data Quality Findings

### Customer Information
- 10,200 raw records representing 10,000 unique customers.
- 3,583 missing Age values (~35% of records).
- 195 exact duplicate rows were identified.
- 200 duplicated CustomerIDs were identified.
- Five duplicated CustomerIDs contained conflicting SignupDate values.
- Age values ranged from 18 to 69, with a median age of 43.
- Gender, Region, and ContractType categories appeared standardized.

### Monthly Usage
- 120,000 records representing 10,000 customers across 12 months.
- Every customer has exactly 12 unique monthly usage records.
- No missing values were identified.
- No exact duplicates were identified.
- CustomerID + Month is unique, confirming one usage record per customer per month.

### Churn Labels
- 10,000 customer-level records.
- No missing values or duplicate CustomerIDs.
- 786 customers churned and 9,214 customers were retained.
- Overall churn rate: 7.86%.

### Cleaning Requirements
1. Remove exact duplicate customer records.
2. Reconcile conflicting SignupDate values for duplicated CustomerIDs using a documented business rule.
3. Investigate the missing Age values before selecting an imputation or treatment strategy.
4. Convert date fields to appropriate datetime formats.
5. Validate customer relationships across all three datasets before producing processed datasets.

## Missing Age Investigation

Before imputing or removing missing Age values, we evaluate whether Age missingness is associated with customer churn.

In [32]:
customer_profile = customers.merge(
    churn,
    on="CustomerID",
    how="left"
)

customer_profile.head()

,CustomerID,Age,Gender,Region,ContractType,MonthlyCharges,SignupDate,Churn
0,CUST_0,NaN,Male,Urban,Yearly,48.18,2021-03-20,0
1,CUST_1,NaN,Female,Suburban,Prepaid,38.94,2020-01-25,0
2,CUST_2,46.0,Female,Rural,Monthly,48.56,2022-08-31,0
3,CUST_3,NaN,Female,Urban,Prepaid,24.22,2018-12-09,0
4,CUST_4,60.0,Male,Rural,Monthly,67.86,2022-12-28,0


In [33]:
customer_profile["Churn"].isnull().sum()

np.int64(0)

In [34]:
customer_profile["AgeMissing"] = customer_profile["Age"].isnull()

In [35]:
customer_profile.groupby("AgeMissing")["Churn"].agg(
    Customers="count",
    Churned="sum",
    ChurnRate="mean"
)

,Customers,Churned,ChurnRate
AgeMissing,,,
False,6617,515,0.077830
True,3583,282,0.078705


Checking: Age by customer characteristics

In [37]:
customers.groupby("Gender")["Age"].agg(
    Count="count",
    Median="median",
    Mean="mean"
)

,Count,Median,Mean
Gender,,,
Female,3265,43.0,43.185911
Male,3352,44.0,43.657220


In [38]:
customers.groupby("Region")["Age"].agg(
    Count="count",
    Median="median",
    Mean="mean"
)

,Count,Median,Mean
Region,,,
Rural,2191,43.0,43.409859
Suburban,2249,44.0,43.756336
Urban,2177,42.0,43.096922


In [39]:
customers.groupby("ContractType")["Age"].agg(
    Count="count",
    Median="median",
    Mean="mean"
)

,Count,Median,Mean
ContractType,,,
Monthly,2638,43.0,43.307430
Prepaid,2654,44.0,43.529766
Yearly,1325,43.0,43.447547


### Missing Age Treatment Decision

Approximately 35% of customer records contain missing Age values. Removing these records would result in substantial information loss.

The churn rate for customers with missing Age (7.87%) was nearly identical to customers with recorded Age (7.78%), indicating that Age missingness was not strongly associated with churn.

Age distributions were also evaluated across Gender, Region, and Contract Type. Median ages ranged only from 42 to 44 across these segments.

**Decision:** Missing Age values will be imputed using the overall median age of 43. An `AgeMissing` indicator will be retained to distinguish imputed values from originally observed values.